In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_csv("air_fryers_clean_brand_year.csv")
df["log_brand_share"] = np.log(df["brand_share"])

print(df[["year", "brand", "brand_share", "log_brand_share", "avg_price", "avg_rating"]].head())

   year       brand  brand_share  log_brand_share   avg_price  avg_rating
0  2019     chefman     0.076015        -2.576826   72.963695    4.434119
1  2019      cosori     0.000730        -7.222964  159.990000    4.581818
2  2019   cuisinart     0.107190        -2.233150  229.465274    4.481312
3  2019        dash     0.199721        -1.610832   55.176333    4.390767
4  2019  gowise usa     0.292186        -1.230364   83.575551    4.552259


In [2]:
feature_cols = [
    "avg_price",
    "avg_rating",
    "compact_share",
    "dual_basket_share",
    "oven_style_share",
    "rotisserie_share",
    "window_share"
]

brand_dummies = pd.get_dummies(df["brand"], prefix="brand", drop_first=True)
year_dummies = pd.get_dummies(df["year"], prefix="year", drop_first=True)

X = pd.concat([df[feature_cols], brand_dummies, year_dummies], axis=1)
X = X.astype(float)

y = df["log_brand_share"].astype(float)

print(X.head())
print("\nX shape:", X.shape)
print("y shape:", y.shape)

    avg_price  avg_rating  compact_share  dual_basket_share  oven_style_share  \
0   72.963695    4.434119       1.000000                0.0          0.780977   
1  159.990000    4.581818       1.000000                0.0          0.090909   
2  229.465274    4.481312       0.993812                0.0          0.889851   
3   55.176333    4.390767       1.000000                0.0          0.973431   
4   83.575551    4.552259       0.999773                0.0          0.129398   

   rotisserie_share  window_share  brand_cosori  brand_cuisinart  brand_dash  \
0          0.243455      0.184119           0.0              0.0         0.0   
1          0.090909      0.000000           1.0              0.0         0.0   
2          0.000000      0.000000           0.0              1.0         0.0   
3          0.000000      0.000000           0.0              0.0         1.0   
4          0.128490      0.000000           0.0              0.0         0.0   

   brand_gowise usa  brand_insta

In [3]:
X_const = sm.add_constant(X)
model = sm.OLS(y, X_const).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:        log_brand_share   R-squared:                       0.763
Model:                            OLS   Adj. R-squared:                  0.600
Method:                 Least Squares   F-statistic:                     4.680
Date:                Tue, 05 May 2026   Prob (F-statistic):           9.24e-05
Time:                        13:58:33   Log-Likelihood:                -36.451
No. Observations:                  50   AIC:                             114.9
Df Residuals:                      29   BIC:                             155.1
Df Model:                          20                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const               -13.3049     16.28

In [4]:
price_coef = model.params["avg_price"]
print("Estimated price coefficient:", price_coef)

Estimated price coefficient: -0.03766765298429452


In [5]:
product_feature_coefs = model.params[
    ["compact_share", "dual_basket_share", "oven_style_share", "rotisserie_share", "window_share"]
].sort_values(ascending=False)

print(product_feature_coefs)

window_share         12.880298
compact_share         9.815304
oven_style_share      1.941774
rotisserie_share     -5.674054
dual_basket_share    -9.509686
dtype: float64


In [6]:
brand_coefs = model.params[model.params.index.str.startswith("brand_")].sort_values(ascending=False)
print(brand_coefs)

brand_cuisinart      6.422436
brand_ninja          5.838705
brand_instant_pot    4.626260
brand_gowise usa     3.938996
brand_oster          3.928074
brand_nuwave         3.544883
brand_cosori         2.551946
brand_ultrean        0.942399
brand_dash           0.176655
dtype: float64


In [7]:
year_coefs = model.params[model.params.index.str.startswith("year_")].sort_values(ascending=False)
print(year_coefs)

year_2020    0.119071
year_2021    0.041900
year_2023   -0.003307
year_2022   -0.098860
dtype: float64


In [8]:
print("R-squared:", model.rsquared)
print("Adjusted R-squared:", model.rsquared_adj)

R-squared: 0.7634539500914357
Adjusted R-squared: 0.6003187432579431


1. What is the estimated price coefficient?

The estimated price coefficient is -0.0377.

2. Is it negative? Why is that important?

Yes, it is negative. That means higher prices are associated with lower brand market share, which is what we would expect in a demand model.

3. Which product features are associated with higher demand?

The features most associated with higher demand are window_share, compact_share, and oven_style_share because they have positive coefficients. Rotisserie_share and dual_basket_share have negative coefficients.

4. Which brand dummy coefficients are largest?

The largest brand dummy coefficients are Cuisinart, Ninja, and Instant Pot. Relative to the omitted brand, those brands are associated with higher demand after controlling for price, rating, product features, and year effects.

5. Which year dummy coefficients are largest?

The largest year dummy coefficients are 2020 and 2021. Relative to the omitted year, those years are associated with higher market share levels on average.

6. What is the model’s R²?

The model’s R² is about 0.763, which means it explains about 76.3% of the variation in log brand share. The adjusted R² is about 0.600.